### Setup
Autoreload and imports

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Imports
import sys
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader

### Define Inputs

In [10]:
## Inputs
checkpoint_name, data_name, save_name = FilenameLoader.rand_pos()

# checkpoint_file= f"../../../saved_models/{checkpoint_name}"
checkpoint_file=f"../../../defenses/data-test/randpos/adv_final_ckpt/advtrained_RandomPos-final.ckpt"
data_file = f"../../../data/{data_name}"
# save_path = f"../final_data/{save_name}"
save_path = f"../adv_trained/randpos"

collapsed = False # True for JSMA and CW
save_clean = True

### Load Model and Data
Load the data and model and run on clean data to check the accuracy/F1.

In [6]:
## Load model and data
# Load original model and ART-wrapper classifier
model, classifier = get_model_classifier(checkpoint_file, collapsed)

# Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [7]:
## Run clean data test
out = clean_data_test(model = model, classifier = classifier, # model information
                x_test=x_test, y_test=y_test, # data information
                checkpoint_file=checkpoint_file, data_file=data_file, # to save in json
                save_path=save_path, filename="clean.json", # saving information
                save_results=save_clean,
                collapsed=collapsed
                ) 

# Check if output passes
print('noWrapper:', 'PASS' if out['noWrapper']['f1'] > 0.8 else 'FAIL')
print('wrapper:', 'PASS' if out['wrapper']['f1'] > 0.8 else 'FAIL')

Saved to ../adv_trained/adv_test/clean.json
noWrapper: PASS
wrapper: PASS


### Adversarial Test
Run adversarial attacks and save the outputted metrics

In [ ]:
# running: constpos

for i in range(21, 31):
    eps = float(i/100)
    adv_test(
        classifier = classifier, # classifier
        x_test=x_test, y_test=y_test, # test data
        checkpoint_file=checkpoint_file, data_file=data_file, # for json
        
        end_index = len(y_test.numpy()),
        path = f"{save_path}/test",
        filename=f"adv_eps_{eps}.json",
        collapsed=collapsed,
        Attack=FastGradientMethod,
        eps=eps, # kwargs
        )

=== Attack: FastGradientMethod, kwargs: {'eps': 0.08} ===
Accuracy:           0.8733
Precision:          0.7077
Recall:             0.9773
F1:                 0.8209
ASR (FNR):          0.0227
False Positive Rate:0.1706
TP=362281, TN=727406, FP=149634, FN=8419
Time elapsed:       17.05s
Saved metrics to ../adv_trained/adv_test/test/adv_eps_0.08.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.09} ===
Accuracy:           0.9916
Precision:          0.9946
Recall:             0.9769
F1:                 0.9857
ASR (FNR):          0.0231
False Positive Rate:0.0023
TP=362149, TN=875066, FP=1974, FN=8551
Time elapsed:       16.08s
Saved metrics to ../adv_trained/adv_test/test/adv_eps_0.09.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.1} ===
Accuracy:           0.9912
Precision:          0.9934
Recall:             0.9770
F1:                 0.9851
ASR (FNR):          0.0230
False Positive Rate:0.0028
TP=362176, TN=874618, FP=2422, FN=8524
Time elapsed:       15.30s
Saved metrics